<img src="https://github.com/pmservice/ai-openscale-tutorials/raw/master/notebooks/images/banner.png" align="left" alt="banner">

# GenAI Batch — End-to-End Monitoring (JDBC)
## IBM Watson OpenScale — IBM Cloud Pak for Data v5.4

This **single-notebook** workflow takes a GenAI model from zero to fully-monitored batch
subscription, backed by **JDBC (DB2 / PostgreSQL)** storage. It combines configuration,
Drift v2 baseline generation, data ingestion, evaluation, and metrics display in one
end-to-end flow.

**High-level flow:**
1. Install dependencies and configure credentials.
2. Create a Detached Prompt Template Asset (DPTA) in the deployment space.
3. Register JDBC storage and watsonx.data Spark engine as Integrated Systems.
4. Download the Drift v2 baseline archive from GitHub and trigger
   `execute_prompt_batch_setup` — creates the subscription, landing/curated/quarantine
   tables, and default monitors.
5. Poll setup to completion and extract subscription + monitor instance IDs.
6. Download the PubMedQA scoring dataset from GitHub and load it into the landing table.
7. Wait for the scheduled curation Spark job to populate `curated_<DPTA_ID>`.
8. Trigger on-demand evaluations (Drift v2 + Model Health).
9. Display metrics.

> **Prerequisites:** Fill in every value in the *Configuration* section before running
> any other cell. Environment variables take precedence over the inline strings.

---

## Contents

1. [Install Dependencies](#1-install)
2. [Configuration](#2-configuration)
3. [Setup Validation](#3-validate-setup)
4. [Create a Detached Prompt Template Asset (DPTA)](#4-dpta)
5. [Register Integrated Systems](#5-integrated-systems)
6. [Trigger Prompt Batch Setup (with Drift v2 Archive)](#6-batch-setup)
7. [Track Batch Setup & Confirm Monitor Instances](#7-batch-setup-status)
8. [Insert Scoring Data into Landing Table](#8-insert-data)
9. [Wait for Scheduled Curation](#9-curation)
10. [Model Health Monitoring](#10-model-health)
11. [Drift v2 Monitoring](#11-drift-v2)


<a id="1-install"></a>
## 1. Install Dependencies

> **Restart the kernel** after this cell finishes, then continue from Section 2.

In [1]:
# ----------------------------------------------------------------------------------------------------
# IBM Confidential
# OCO Source Materials
# 5737-H76
# Copyright IBM Corp. 2026
# The source code for this Notebook is not published or otherwise divested of its trade
# secrets, irrespective of what has been deposited with the U.S. Copyright Office.
# ----------------------------------------------------------------------------------------------------

import json
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

VERSION = "genai-batch-e2e-jdbc-1.0"
print("Notebook version:", VERSION)

Notebook version: genai-batch-e2e-jdbc-1.0


In [ ]:
import warnings
warnings.filterwarnings("ignore")
%env PIP_DISABLE_PIP_VERSION_CHECK=1

%pip install -U 'ibm-watson-openscale~=3.1.11' 'ibm-wos-utils~=5.4.2' 'ibm-watsonx-ai~=1.8' --no-cache | tail -n 1
%pip install -U python-dotenv 'ibm_db>=3.2.3,<3.2.7' --no-cache | tail -n 1

# PostgreSQL only: uncomment the line below when JDBC_URL points to a PostgreSQL database.
# %pip install -U 'psycopg2-binary~=2.9.9' --no-cache | tail -n 1


<a id="2-configuration"></a>
## 2. Configuration

All environment-specific values are defined in this section — **nothing below it needs editing**.
Values are read from environment variables first (useful for CI or `.env` files), with the
inline strings as interactive fallbacks.

> **Tip:** Copy `.env.example` (next to this notebook) to `test.env` and fill in your
> values. The first cell below loads `test.env` automatically. In a CPD project, upload
> `test.env` as a project asset and uncomment the download lines in that cell.

### 2-A. Load environment file

In [3]:
import os
from dotenv import load_dotenv

# Running in a CPD project? A file uploaded with Project -> Assets -> Upload asset is not
# placed in the notebook's working directory. Uncomment the lines below to download
# test.env from the project before it is loaded.
# from ibm_watson_studio_lib import access_project_or_space
# wslib = access_project_or_space()
# wslib.download_file("test.env", "/home/wsuser/work/test.env")

# Loads variables from test.env (if present) without overriding anything
# already set in the shell environment.
load_dotenv("test.env", override=False)


True

### 2-B. CPD credentials

In [4]:
CPD_URL      = os.environ.get("CPD_URL",      "https://<cpd-host>")   # EDIT THIS
CPD_USERNAME = os.environ.get("CPD_USERNAME", "<username>")           # EDIT THIS
CPD_PASSWORD = os.environ.get("CPD_PASSWORD", "<password>")           # EDIT THIS
CPD_APIKEY   = os.environ.get("CPD_APIKEY",   "<apikey>")             # EDIT THIS

SPACE_ID            = os.environ.get("SPACE_ID",            "<deployment-space-id>")     # EDIT THIS
SERVICE_INSTANCE_ID = os.environ.get("SERVICE_INSTANCE_ID", "00000000-0000-0000-0000-000000000000")  # EDIT THIS


### 2-C. Model and DPTA details

| Variable | Description |
| :- | :- |
| `MODEL_ID` | Full model identifier used internally (e.g. `google/flan-t5-xl`) |
| `MODEL_NAME` | Short human-readable name shown in the UI (e.g. `flan-t5-xl`) |
| `MODEL_PROVIDER` | Name of the model provider (e.g. `Hugging Face`) |
| `MODEL_URL` | URL of the model on the provider's platform |
| `PROMPT_URL` | URL identifying the prompt template |
| `PROMPT_INPUT` | Input template string — must reference `QUESTION_FIELD` and `CONTEXT_FIELDS` |
| `DPTA_NAME` | Asset name shown in the CPD deployment space |
| `LABEL_COLUMN` | Ground-truth column name in your scoring data |
| `TASK_ID` | GenAI task type (`retrieval_augmented_generation`, `summarization`, …) |

In [5]:
MODEL_ID       = os.environ.get("MODEL_ID",       "google/flan-t5-xl")                            # EDIT THIS
MODEL_NAME     = os.environ.get("MODEL_NAME",     "flan-t5-xl")                                   # EDIT THIS
MODEL_PROVIDER = os.environ.get("MODEL_PROVIDER", "Hugging Face")                                 # EDIT THIS
MODEL_URL      = os.environ.get("MODEL_URL",      "https://huggingface.co/google/flan-t5-xl")     # EDIT THIS
PROMPT_URL     = os.environ.get("PROMPT_URL",     "<prompt-url>")                                 # EDIT THIS

# Must reference the variable names in CONTEXT_FIELDS and QUESTION_FIELD below.
PROMPT_INPUT   = os.environ.get("PROMPT_INPUT",   "{question} {context}")                         # EDIT THIS

DPTA_NAME    = os.environ.get("DPTA_NAME",    "External prompt sample for genAI batch JDBC")      # EDIT THIS
LABEL_COLUMN = os.environ.get("LABEL_COLUMN", "ground_truth")
TASK_ID      = os.environ.get("TASK_ID",      "retrieval_augmented_generation")  # RAG | summarization | question_answering


### 2-D. JDBC storage

| Variable | Description |
| :- | :- |
| `JDBC_DATABASE_NAME` | Target database name (e.g. `SAMPLE` for DB2) |
| `JDBC_SCHEMA_NAME` | Target schema (e.g. `GENAI_BATCH`) |
| `JDBC_URL` | JDBC connection URL: `jdbc:db2://<host>:<port>/<db>` or `jdbc:postgresql://<host>:<port>/<db>`. Do not add `:sslConnection=true;`; SSL is controlled by `JDBC_USE_SSL` |
| `JDBC_DB_DRIVER` | JDBC driver class |
| `JDBC_USE_SSL` | SSL is on by default; set `"false"` for a database without SSL |
| `JDBC_CERT` | Base64-encoded SSL certificate of the database server. Needed with SSL unless the certificate is signed by a public CA; leave blank when SSL is off |

In [6]:
JDBC_DATABASE_NAME = os.environ.get("JDBC_DATABASE_NAME", "SAMPLE")   # EDIT THIS
JDBC_SCHEMA_NAME   = os.environ.get("JDBC_SCHEMA_NAME",   "<schema>") # EDIT THIS

# DB2 example:        "jdbc:db2://<host>:50001/<DBNAME>"
# Do not add ":sslConnection=true;" to the URL. SSL is controlled by JDBC_USE_SSL.
# PostgreSQL example: "jdbc:postgresql://<host>:5432/<DBNAME>"
# Without SSL (JDBC_USE_SSL=false), DB2: "jdbc:db2://<host>:50000/<DBNAME>"
JDBC_URL       = os.environ.get("JDBC_URL",       "jdbc:db2://<host>:50001/SAMPLE")  # EDIT THIS

# DB2: "com.ibm.db2.jcc.DB2Driver"   PostgreSQL: "org.postgresql.Driver"
JDBC_DB_DRIVER = os.environ.get("JDBC_DB_DRIVER", "com.ibm.db2.jcc.DB2Driver")

JDBC_USE_SSL   = os.environ.get("JDBC_USE_SSL", "true").lower() == "true"
JDBC_CERT      = os.environ.get("JDBC_CERT",    "")  # Base64-encoded SSL cert; needed unless the cert is signed by a public CA

JDBC_USERNAME  = os.environ.get("JDBC_USERNAME", "<db-username>")  # EDIT THIS
JDBC_PASSWORD  = os.environ.get("JDBC_PASSWORD", "<db-password>")  # EDIT THIS


### 2-E. watsonx.data Spark engine

| Variable | Description |
| :- | :- |
| `LAKEHOUSE_INSTANCE_ID` | ID of your watsonx.data (lakehouse) instance. The Spark engine's applications URL is looked up from it, so you don't set it. |
| `SPARK_VOLUME` | CPD storage volume the Spark job mounts, e.g. `cpd-instance::MyVolume` (Storage volumes page). Must **not** be the Spark engine's own home volume — setup validation checks this and lists the volumes you can use. |
| `SPARK_ENGINE_ID` | *(optional)* e.g. `spark362`. Only needed when the instance has more than one Spark engine. |

> **Note:** Starting with CPD 5.4 the **IBM Analytics Engine (IAE)** is deprecated.
> Use the **watsonx.data Spark** engine of your lakehouse instance.

In [7]:
LAKEHOUSE_INSTANCE_ID = os.environ.get("LAKEHOUSE_INSTANCE_ID", "<EDIT THIS>")  # EDIT THIS
SPARK_VOLUME         = os.environ.get("SPARK_VOLUME", "<EDIT THIS>")  # EDIT THIS
SPARK_ENGINE_ID       = os.environ.get("SPARK_ENGINE_ID", "")  # optional: only if the instance has several Spark engines

# Integrated system display names (used as idempotency keys — change only if you want different names).
JDBC_CONNECTION_NAME        = os.environ.get("JDBC_CONNECTION_NAME",        "JDBC_CONNECTION")
JDBC_CONNECTION_DESCRIPTION = os.environ.get("JDBC_CONNECTION_DESCRIPTION", "")
SPARK_ENGINE_NAME           = os.environ.get("SPARK_ENGINE_NAME",           "Spark Engine - GenAI Batch")
SPARK_ENGINE_DESCRIPTION    = os.environ.get("SPARK_ENGINE_DESCRIPTION",    "")
SPARK_DISPLAY_NAME          = os.environ.get("SPARK_DISPLAY_NAME",          "WXD Spark - GenAI Batch")


### 2-F. Prompt variable names

These must match the variable names defined in your DPTA prompt template.

| Variable | Description |
| :- | :- |
| `CONTEXT_FIELDS` | List with **exactly one** element — the prompt variable carrying retrieved context (RAG) |
| `QUESTION_FIELD` | Prompt variable holding the user question |

In [8]:
CONTEXT_FIELDS = ["context"]  # EDIT THIS — must be a list with exactly one element
QUESTION_FIELD = "question"   # EDIT THIS

<a id="3-validate-setup"></a>
## 3. Setup Validation

Validates every configuration value **before** any API call and aggregates all problems
into a single error so you see them all at once.

In [9]:
import warnings
import urllib3
warnings.filterwarnings("ignore")
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

from ibm_cloud_sdk_core.authenticators import CloudPakForDataAuthenticator
from ibm_watson_openscale import APIClient
from ibm_watson_openscale.utils.gen_ai_batch_utility import (
    validate_setup,
    validate_context_fields,
)

authenticator = CloudPakForDataAuthenticator(
    url=CPD_URL,
    username=CPD_USERNAME,
    password=CPD_PASSWORD,
    disable_ssl_verification=True,
)
wos_client = APIClient(
    authenticator=authenticator,
    service_url=CPD_URL,
    service_instance_id=SERVICE_INSTANCE_ID,
)

# Look up the Spark engine's applications URL and home volume from the lakehouse instance.
from ibm_watson_openscale.utils.gen_ai_batch_utility import discover_spark_engine
_spark = discover_spark_engine(
    wos_client=wos_client,
    cpd_url=CPD_URL,
    lakehouse_instance_id=LAKEHOUSE_INSTANCE_ID,
    spark_engine_id=SPARK_ENGINE_ID or None,
    disable_ssl_verification=True,
)
SPARK_ENDPOINT = _spark["spark_endpoint"]
SPARK_HOME_VOLUME = _spark["home_volume"]  # not usable as SPARK_VOLUME; validate_setup checks

_config = {
    "CPD_URL":            CPD_URL,
    "CPD_USERNAME":       CPD_USERNAME,
    "CPD_PASSWORD":       CPD_PASSWORD,
    "CPD_APIKEY":         CPD_APIKEY,
    "SPACE_ID":           SPACE_ID,
    "JDBC_DATABASE_NAME": JDBC_DATABASE_NAME,
    "JDBC_SCHEMA_NAME":   JDBC_SCHEMA_NAME,
    "JDBC_URL":           JDBC_URL,
    "JDBC_DB_DRIVER":     JDBC_DB_DRIVER,
    "JDBC_USERNAME":      JDBC_USERNAME,
    "JDBC_PASSWORD":      JDBC_PASSWORD,
    "JDBC_USE_SSL":       JDBC_USE_SSL,
    "JDBC_CERT":          JDBC_CERT,
    "SPARK_ENDPOINT":     SPARK_ENDPOINT,
    "SPARK_VOLUME":       SPARK_VOLUME,
    "SPARK_HOME_VOLUME":  SPARK_HOME_VOLUME,
    "LAKEHOUSE_INSTANCE_ID": LAKEHOUSE_INSTANCE_ID,
}
validate_setup(_config, wos_client, disable_ssl_verification=True)
validate_context_fields(CONTEXT_FIELDS)


Spark engine spark468 (WXDBatchSpark), home volume cpd-instance::wxdInstVol


✅ Authentication OK. wos_client version: 1.0.0


✅ JDBC connection OK; schema 'PPG' exists.


⚠️  WARNING: LLM / GenAI batch subscription targeting a Db2 datamart. Ensure your watsonx.governance instance is NOT using Db2 as its internal datamart — that is a separate concern from the JDBC landing tables.
✅ Setup validated — all checks OK.
✅ CONTEXT_FIELDS validated: ['context']


### 3-A. Map the deployment space to the OpenScale instance

On Cloud Pak for Data the deployment space must be mapped to the OpenScale service instance
before a subscription can be created in it. The call is idempotent — an existing mapping is
left as it is.

In [10]:
wos_client.wos.add_instance_mapping(
    service_instance_id=SERVICE_INSTANCE_ID,
    space_id=SPACE_ID,
)


Instance mapping already exists. Skipping creation.


{'instance_mappings': [{'entity': {'service_instance_id': '00000000-0000-0000-0000-000000000000',
    'status': {'state': 'active'},
    'target': {'target_id': 'cb29fbe8-6c2a-4b9c-8997-1951946f0ccf',
     'target_type': 'space'}},
   'metadata': {'account_id': '',
    'created_at': '2026-10-12T06:32:04.788Z',
    'created_by': 'ibm',
    'crn': 'crn:v1:bluemix:public:aiopenscale:unknown:a/:ibm:instance_mapping:01a12ecf-adb1-7ba4-a3ad-92f8a2f6d816',
    'id': '01a12ecf-adb1-7ba4-a3ad-92f8a2f6d816',
    'url': '/v2/instance_mappings/01a12ecf-adb1-7ba4-a3ad-92f8a2f6d816'}}]}

<a id="4-dpta"></a>
## 4. Create a Detached Prompt Template Asset (DPTA)

A DPTA represents an externally-hosted model in the CPD catalog. We create it **directly in
the deployment space** using the `ibm-watsonx-ai` SDK.

> **Reuse:** Step 4-B reuses a DPTA of the same name only if it has no subscription yet.
> One that is already monitored cannot go through batch setup again, so a new DPTA is created.

### 4-A. Initialise the prompt template manager

In [11]:
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models.prompts import (
    DetachedPromptTemplate,
    PromptTemplateManager,
)

wx_credentials = Credentials(
    url=CPD_URL,
    username=CPD_USERNAME,
    api_key=CPD_APIKEY,
    instance_id="openshift",
    version="5.4",
    verify=False,
)
prompt_mgr = PromptTemplateManager(credentials=wx_credentials, space_id=SPACE_ID)
print("✅ Prompt template manager initialised.")


✅ Prompt template manager initialised.


### 4-B. Create (or reuse) DPTA

In [12]:
from ibm_watson_openscale.utils.gen_ai_batch_utility import find_detached_prompts

# Reuse a DPTA of the same name only when it is not monitored yet (for example one left
# by an interrupted run). A DPTA that already has a subscription cannot go through batch
# setup again, so a new one is created instead.
DPTA_ID = None
_existing = find_detached_prompts(
    wos_client=wos_client,
    cpd_url=CPD_URL,
    space_id=SPACE_ID,
    name=DPTA_NAME,
    disable_ssl_verification=True,
)
_unmonitored = [p for p in _existing if not p["subscription_id"]]
if _unmonitored:
    DPTA_ID = _unmonitored[0]["asset_id"]
    print(f"ℹ️  DPTA '{DPTA_NAME}' exists without a subscription — reusing DPTA_ID: {DPTA_ID}")
elif _existing:
    print(f"ℹ️  {len(_existing)} DPTA(s) named '{DPTA_NAME}' already have a subscription — creating a new one.")

if DPTA_ID is None:
    prompt_template = DetachedPromptTemplate(
        name=DPTA_NAME,
        description="Detached prompt for GenAI batch monitoring",
        model_id=MODEL_ID,
        task_ids=[TASK_ID],
        input_text=PROMPT_INPUT,
        input_variables={
            QUESTION_FIELD:    {"default_value": ""},
            CONTEXT_FIELDS[0]: {"default_value": ""},
        },
        input_prefix="Input:",
        output_prefix="Output:",
        detached_prompt_id="detached_prompt",
        detached_model_id=MODEL_ID,
        detached_model_provider=MODEL_PROVIDER,
        detached_model_name=MODEL_NAME,
        detached_model_url=MODEL_URL,
        detached_prompt_url=PROMPT_URL,
        detached_prompt_additional_information=[
            {"model_owner": MODEL_PROVIDER.lower().replace(" ", "")}
        ],
    )
    stored_prompt_template = prompt_mgr.store_prompt(prompt_template=prompt_template)
    DPTA_ID = stored_prompt_template.prompt_id
    # A stored prompt is locked by its creator; unlock it so batch setup can use it.
    prompt_mgr.unlock(prompt_id=DPTA_ID)
    print("✅ Detached PTA created in space. DPTA_ID:", DPTA_ID)

assert DPTA_ID, "DPTA_ID is not set — creation or lookup failed. Check the output above."


✅ Detached PTA created in space. DPTA_ID: 01a135d5-b456-7360-9933-851951b744d9


<a id="5-integrated-systems"></a>
## 5. Register Integrated Systems

Registers the JDBC storage and the watsonx.data Spark engine as Integrated Systems in
watsonx.governance. Both registrations are **idempotent** — an existing system of the same
name is reused. The returned IDs are required by the batch setup call in Section 6.

> **Note:** Starting with CPD 5.4, the **IBM Analytics Engine (IAE)** is deprecated.
> Use the **watsonx.data Spark** engine endpoint (`/lakehouse/api/v3/spark_engines/…/applications`).

### 5-A. Register JDBC storage

In [13]:
from ibm_watson_openscale.utils.gen_ai_batch_utility import register_jdbc_storage

jdbc_integrated_system_id = register_jdbc_storage(
    wos_client=wos_client,
    name=JDBC_CONNECTION_NAME,
    description=JDBC_CONNECTION_DESCRIPTION,
    jdbc_url=JDBC_URL,
    jdbc_username=JDBC_USERNAME,
    jdbc_password=JDBC_PASSWORD,
    jdbc_db_driver=JDBC_DB_DRIVER,
    jdbc_use_ssl=JDBC_USE_SSL,
    jdbc_cert=JDBC_CERT,
)


Registering JDBC storage integrated system...


✅ Integrated system 'DB2_BATCH_SSL_NOSUFFIX' created. ID: 01a135d5-bc37-7ab2-9a33-646411a2cb57
JDBC storage integrated system ID: 01a135d5-bc37-7ab2-9a33-646411a2cb57


### 5-B. Register watsonx.data Spark engine

In [14]:
from ibm_watson_openscale.utils.gen_ai_batch_utility import register_spark_engine

spark_integrated_system_id = register_spark_engine(
    wos_client=wos_client,
    name=SPARK_ENGINE_NAME,
    description=SPARK_ENGINE_DESCRIPTION,
    cpd_apikey=CPD_APIKEY,
    cpd_username=CPD_USERNAME,
    spark_endpoint=SPARK_ENDPOINT,
    spark_volume=SPARK_VOLUME,
    display_name=SPARK_DISPLAY_NAME,
)


Registering watsonx.data Spark integrated system...


ℹ️  Integrated system 'Spark Engine - GenAI Batch' already exists — reusing ID: 01a12ed2-d688-7241-afb1-06e3ee517a5f
Spark integrated system ID: 01a12ed2-d688-7241-afb1-06e3ee517a5f


<a id="6-batch-setup"></a>
## 6. Trigger Prompt Batch Setup (with Drift v2 Archive)

`execute_prompt_batch_setup` creates the landing, curated and quarantine tables in the JDBC
database, creates the watsonx.governance subscription and its default monitor instances, and
schedules an hourly curation Spark job that moves records from landing to curated and
quarantine. `background_mode=True` returns immediately (HTTP 202); Section 7 polls for
completion.


### 6-A — Download Drift v2 baseline archive from GitHub

> **This notebook uses `train_archive = False`** in the Drift v2 monitor parameters: a
> pre-built baseline archive is downloaded here and supplied to batch setup.
>
> It does not cover `train_archive = True`. In that case no archive is supplied and Drift v2
> builds its baseline from your own records, so the monitor goes to `error` if it is
> activated before any records are loaded. The flow is then:
> 1. Add the baseline records to the landing table **before** the monitors are active.
> 2. Wait for curation.
> 3. Wait for the Drift v2 configuration to finish (the monitor becomes `active`).
> 4. Upload the records to evaluate.
> 5. Wait for curation again.
> 6. Run the evaluation and wait for it to finish.

In [15]:
import os, urllib.request

ARCHIVE_URL  = (
    "https://raw.githubusercontent.com/IBM/watson-openscale-samples/main/"
    "cpd_5.4/Gen%20AI/configuration/batch/assets/baseline_archive.tar.gz"
)
# Resolve absolute path so drift_archive= always points to the downloaded file.
NOTEBOOK_DIR = os.getcwd()
ARCHIVE_PATH = os.path.join(NOTEBOOK_DIR, "baseline_archive.tar.gz")

print(f"Downloading to {ARCHIVE_PATH} ...")
urllib.request.urlretrieve(ARCHIVE_URL, ARCHIVE_PATH)
print(f"✅ Downloaded ({os.path.getsize(ARCHIVE_PATH):,} bytes)")


✅ Downloaded (26,091 bytes)


## ⚠️ STOP HERE — this step is not reversible

It creates tables and a subscription that re-running will **not** overwrite cleanly.
Before continuing, confirm the JDBC database and schema exist and that the database user
holds `CREATE TABLE`. To re-run later, delete the existing subscription in the
watsonx.governance UI first.

### 6-B — Trigger batch setup

In [16]:
SUPPORTING_MONITORS = {
    "model_health": {},
    "drift_v2": {
        "parameters": {
            "train_archive":        False,  # use the pre-built archive from step 6-A
            "baseline_max_samples": 10000,
        }
    },
}

batch_setup_response = wos_client.wos.execute_prompt_batch_setup(
    prompt_template_asset_id=DPTA_ID,
    label_column=LABEL_COLUMN,
    operational_space_id="production",
    problem_type=TASK_ID,
    input_data_type="unstructured_text",
    space_id=SPACE_ID,
    analytics_engine={"integrated_system_id": spark_integrated_system_id},
    data_sources=[
        {
            "integrated_system_id": jdbc_integrated_system_id,
            "database_name":        JDBC_DATABASE_NAME,
            "schema_name":          JDBC_SCHEMA_NAME,
        }
    ],
    context_fields=CONTEXT_FIELDS,
    question_field=QUESTION_FIELD,
    supporting_monitors=SUPPORTING_MONITORS,
    background_mode=True,
    drift_archive=ARCHIVE_PATH,
)

print("execute_prompt_batch_setup status:", batch_setup_response.status_code)
print(json.dumps(batch_setup_response.result.to_dict(), indent=2))


execute_prompt_batch_setup status: 202
{
  "prompt_template_asset_id": "01a135d5-b456-7360-9933-851951b744d9",
  "space_id": "cb29fbe8-6c2a-4b9c-8997-1951946f0ccf",
  "start_time": "2026-10-13T15:16:04.985598Z",
  "status": {
    "state": "RUNNING"
  }
}


<a id="7-batch-setup-status"></a>
## 7. Track Batch Setup & Confirm Monitor Instances

Polls `get_prompt_batch_setup` until the state leaves `RUNNING`, then extracts the
subscription ID and all monitor instance IDs from the final response.
Raises on `ERROR` or timeout.

In [17]:
from ibm_watson_openscale.utils.gen_ai_batch_utility import poll_and_extract_batch_setup

batch_setup_poll_response = poll_and_extract_batch_setup(
    wos_client=wos_client,
    dpta_id=DPTA_ID,
    space_id=SPACE_ID,
)

SUBSCRIPTION_ID              = batch_setup_poll_response["subscription_id"]
mhm_monitor_instance_id      = batch_setup_poll_response["mhm_monitor_instance_id"]
drift_v2_monitor_instance_id = batch_setup_poll_response["drift_v2_monitor_instance_id"]
mrm_monitor_instance_id      = batch_setup_poll_response["mrm_monitor_instance_id"]


20:46:06 state: RUNNING
{
  "prompt_template_asset_id": "01a135d5-b456-7360-9933-851951b744d9",
  "space_id": "cb29fbe8-6c2a-4b9c-8997-1951946f0ccf",
  "start_time": "2026-10-13T15:16:04.985598Z",
  "status": {
    "state": "RUNNING"
  }
}


20:46:37 state: RUNNING
{
  "prompt_template_asset_id": "01a135d5-b456-7360-9933-851951b744d9",
  "space_id": "cb29fbe8-6c2a-4b9c-8997-1951946f0ccf",
  "start_time": "2026-10-13T15:16:04.985598Z",
  "status": {
    "state": "RUNNING"
  }
}


20:47:08 state: RUNNING
{
  "prompt_template_asset_id": "01a135d5-b456-7360-9933-851951b744d9",
  "space_id": "cb29fbe8-6c2a-4b9c-8997-1951946f0ccf",
  "start_time": "2026-10-13T15:16:04.985598Z",
  "status": {
    "state": "RUNNING"
  }
}


20:47:39 state: FINISHED
{
  "prompt_template_asset_id": "01a135d5-b456-7360-9933-851951b744d9",
  "space_id": "cb29fbe8-6c2a-4b9c-8997-1951946f0ccf",
  "service_provider_id": "01a12ed2-e0f0-76b5-a694-e886c52f24c8",
  "subscription_id": "01a135d6-bcfe-7eff-9b96-a26159098581",
  "start_time": "2026-10-13T15:16:04.985598Z",
  "end_time": "2026-10-13T15:17:15.640264Z",
  "status": {
    "state": "FINISHED"
  },
  "monitors": [
    {
      "monitor_definition_id": "model_health",
      "monitor_instance_id": "01a135d6-bf53-764f-8d3e-83f57ac5dd2a"
    },
    {
      "monitor_definition_id": "drift_v2",
      "monitor_instance_id": "01a135d6-c5ea-717a-a1a9-9442f1c57520"
    },
    {
      "monitor_definition_id": "mrm",
      "monitor_instance_id": "01a135d6-c88c-7d18-aee7-20799e757f29"
    }
  ],
  "evaluation_tests": [
    "model_health",
    "drift_v2"
  ]
}

SUBSCRIPTION_ID             : 01a135d6-bcfe-7eff-9b96-a26159098581
mhm_monitor_instance_id     : 01a135d6-bf53-764f-8d3e-83f57ac5dd

### 7-B. Confirm the monitors are active

Checked here, before any data is loaded, so a monitor that failed to configure is found now
and not after the curation wait in Section 9. Monitors become `active` without curated data.

In [18]:
from ibm_watson_openscale.utils.gen_ai_batch_utility import wait_for_monitor_active

for _name, _instance_id in (
    ("model_health", mhm_monitor_instance_id),
    ("drift_v2", drift_v2_monitor_instance_id),
):
    if not _instance_id:
        print(f"⚠️  No {_name} monitor instance was created — its section below will be skipped.")
        continue
    wait_for_monitor_active(
        wos_client=wos_client,
        monitor_instance_id=_instance_id,
        monitor_name=_name,
    )

Waiting for 'model_health' (01a135d6-bf53-764f-8d3e-83f57ac5dd2a) to become active ...


  [20:47:40] attempt  1/30 — model_health: active
  ✅ model_health is active.
Waiting for 'drift_v2' (01a135d6-c5ea-717a-a1a9-9442f1c57520) to become active ...


  [20:47:40] attempt  1/30 — drift_v2: active
  ✅ drift_v2 is active.


<a id="8-insert-data"></a>
## 8. Insert Scoring Data into Landing Table

Before triggering evaluations, load your scoring records into the
**`landing_<DPTA_ID>`** table that was created by the batch setup.
The curation job (running on Spark) will then move records from the landing table to
the **`curated_<DPTA_ID>`** table, from which monitor evaluations read.

### Landing table schema

| Column | Type | Nullable | Notes |
| :- | :- | :- | :- |
| `scoring_id` | VARCHAR(128) | YES | Unique identifier for the scoring request |
| `scoring_timestamp` | TIMESTAMP | YES | When the model was called |
| `response_time` | INTEGER | YES | Model response time in milliseconds |
| `user_id` | VARCHAR(128) | YES | Identifier of the user who made the request |
| `error_code` | BIGINT | YES | Error code if the request failed; `null` on success |
| `request` | CLOB(1 MB) | YES | Raw model input / prompt text |
| `response` | CLOB(1 MB) | YES | Raw model output / generated text |
| `context` | CLOB(1 MB) | YES | Retrieved context (RAG) or additional metadata |
| `ground_truth` | CLOB(1 MB) | YES | Reference / expected output for quality evaluation |
| `ingest_timestamp` | TIMESTAMP | **NOT NULL** | Auto-set to `CURRENT TIMESTAMP` on insert (via trigger) |
| `_ingest_seq` | BIGINT | **NOT NULL** | Auto-incremented IDENTITY — do not supply on insert |

#### Sample data

| `scoring_id` | `scoring_timestamp` | `response_time` | `user_id` | `error_code` | `request` | `response` | `context` | `ground_truth` |
| :- | :- | :- | :- | :- | :- | :- | :- | :- |
| def456-001 | 2026-08-02 10:05:12 | 580 | user_1 | null | `<prompt text>` | `<generated text>` | `<context>` | `<reference answer>` |
| ghi789-001 | 2026-08-02 10:10:33 | 200 | user_2 | 500 | `<prompt text>` | `<generated text>` | `<context>` | `<reference answer>` |
| abc123-001 | 2026-08-01 10:00:05 | 320 | user_3 | null | `<prompt text>` | `<generated text>` | `<context>` | `<reference answer>` |

> **Note:** Do **not** include `ingest_timestamp` or `_ingest_seq` in your `INSERT`
> statements — they are populated automatically by the trigger and the IDENTITY column.


Load full dataset using the method appropriate for your environment
(direct SQL, ibm_db, JDBC, pandas + SQLAlchemy, etc.).
Once records are inserted, proceed to Section 9 to trigger curation.

### Ingest pubmedqa_jsonl.zip into `landing_<DPTA_ID>`

#### 8-A — Download pubmedqa_jsonl.zip

In [19]:
import os, urllib.request

ZIP_URL  = (
    "https://raw.githubusercontent.com/IBM/watson-openscale-samples/main/"
    "cpd_5.4/Gen%20AI/configuration/batch/assets/pubmedqa_jsonl.zip"
)
NOTEBOOK_DIR = os.getcwd()
ZIP_PATH     = os.path.join(NOTEBOOK_DIR, "pubmedqa_jsonl.zip")

print(f"Downloading to {ZIP_PATH} ...")
urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
print(f"✅ Downloaded ({os.path.getsize(ZIP_PATH):,} bytes)")


✅ Downloaded (30,495,699 bytes)


#### 8-B — Ingest PubMedQA data into `landing_<DPTA_ID>`

Parses the zip archive, opens a Db2 or PostgreSQL connection, batch-inserts all rows into
the landing table, and validates the row count — all in one call.

In [20]:
from ibm_watson_openscale.utils.gen_ai_batch_utility import ingest_scoring_data_to_landing_jdbc

total_written = ingest_scoring_data_to_landing_jdbc(
    zip_path=ZIP_PATH,
    jdbc_url=JDBC_URL,
    jdbc_username=JDBC_USERNAME,
    jdbc_password=JDBC_PASSWORD,
    jdbc_use_ssl=JDBC_USE_SSL,
    jdbc_cert=JDBC_CERT,
    schema_name=JDBC_SCHEMA_NAME,
    dpta_id=DPTA_ID,
)
print(f"Total rows written to landing table: {total_written:,}")


Loaded 10,000 raw records from pubmedqa_jsonl.zip


Parsed 10,000 landing rows from 10,000 line(s) — 0 unmappable


✅ DB2 connection established
Writing 10,000 rows to PPG.landing_01a135d5-b456-7360-9933-851951b744d9 ...


  10,000 / 10,000
✅ 10,000 rows inserted into PPG.landing_01a135d5-b456-7360-9933-851951b744d9


DB COUNT(*) = 10,000
✅ 10,000 rows confirmed in PPG.landing_01a135d5-b456-7360-9933-851951b744d9 — proceed to next section.
Total rows written to landing table: 10,000


#### 8-C — Object Storage variant

> **For Object Storage notebooks:** replace step 8-B with the corresponding
> `ingest_scoring_data_to_landing_object_storage()` utility function from
> `ibm_watson_openscale.utils.gen_ai_batch_utility`. It takes the same
> `zip_path` and `dpta_id` arguments, with the Object Storage credentials
> (`cos_access_key_id`, `cos_secret_access_key`, `cos_endpoint`,
> `cos_bucket_name`, `cos_warehouse_uri`, `service_instance_id`) in place of
> the JDBC connection parameters and `schema_name`.

<a id="9-curation"></a>
## 9. Wait for Scheduled Curation

The curation Spark job runs on a **fixed schedule** (default: hourly). It reads all new
rows from `landing_<DPTA_ID>`, validates them, and moves valid rows to `curated_<DPTA_ID>`
and invalid rows to `quarantine_<DPTA_ID>`.

> **There is no way to skip this wait.** The evaluations in Sections 10 and 11 read from
> `curated_<DPTA_ID>`, so the curation job must complete at least one cycle before
> proceeding.

The cell below runs `COUNT(*)` on the landing, curated and quarantine tables every 60 seconds
and reports the state of the subscription's curation Spark job (`curation_job_<SUBSCRIPTION_ID>`).
It blocks until the curated and quarantine rows add up to the landed rows. It stops early with
an error if the Spark job fails or every row is quarantined. The first job runs about one hour
after the subscription was created.

In [21]:
from ibm_watson_openscale.utils.gen_ai_batch_utility import wait_for_curation_jdbc

curated_count = wait_for_curation_jdbc(
    jdbc_url=JDBC_URL,
    jdbc_username=JDBC_USERNAME,
    jdbc_password=JDBC_PASSWORD,
    jdbc_use_ssl=JDBC_USE_SSL,
    jdbc_cert=JDBC_CERT,
    schema_name=JDBC_SCHEMA_NAME,
    dpta_id=DPTA_ID,
    # Also watch the subscription's curation Spark job, so a failed job ends the wait at once.
    wos_client=wos_client,
    spark_endpoint=SPARK_ENDPOINT,
    lakehouse_instance_id=LAKEHOUSE_INSTANCE_ID,
    subscription_id=SUBSCRIPTION_ID,
    disable_ssl_verification=True,
)
print(f"Curated row count: {curated_count:,}")


Waiting for the scheduled curation job to process PPG.landing_01a135d5-b456-7360-9933-851951b744d9 ...
Polling every 60s — max wait ~120 min.


  [20:52:46] attempt  1/120 — landing: 10,000, curated: 0, quarantine: 0 — curation job: not submitted yet


  [20:53:53] attempt  2/120 — landing: 10,000, curated: 0, quarantine: 0 — curation job: not submitted yet


  [20:54:59] attempt  3/120 — landing: 10,000, curated: 0, quarantine: 0 — curation job: not submitted yet


  [20:56:06] attempt  4/120 — landing: 10,000, curated: 0, quarantine: 0 — curation job: not submitted yet


...

  [21:47:15] attempt 50/120 — landing: 10,000, curated: 0, quarantine: 0 — curation job: not submitted yet


  [21:48:21] attempt 51/120 — landing: 10,000, curated: 0, quarantine: 0 — curation job: RUNNING


  [21:49:29] attempt 52/120 — landing: 10,000, curated: 0, quarantine: 0 — curation job: RUNNING


  [21:50:36] attempt 53/120 — landing: 10,000, curated: 0, quarantine: 0 — curation job: RUNNING


  [21:51:45] attempt 54/120 — landing: 10,000, curated: 10,000, quarantine: 0 — curation job: FINISHED

✅ Curation complete — 10,000 curated, 0 quarantined (of 10,000 landed). Proceeding to next section.
Curated row count: 10,000


<a id="10-model-health"></a>
## 10. Model Health Monitoring

Confirms the monitor is `active`, triggers an on-demand evaluation run, polls until
the run finishes, and displays metrics.

> **Before running:** Ensure the curation job has completed and records are
> present in the `curated_<DPTA_ID>` table (confirmed in Section 9).

### 10-A. Re-check Model Health monitor is active

In [22]:
from ibm_watson_openscale.utils.gen_ai_batch_utility import wait_for_monitor_active

if not mhm_monitor_instance_id:
    print("⚠️  mhm_monitor_instance_id is not set — skipping Model Health.")
else:
    wait_for_monitor_active(
        wos_client=wos_client,
        monitor_instance_id=mhm_monitor_instance_id,
        monitor_name="model_health",
    )


Waiting for 'model_health' (01a135d6-bf53-764f-8d3e-83f57ac5dd2a) to become active ...


  [21:51:46] attempt  1/30 — model_health: active
  ✅ model_health is active.


### 10-B. Run Model Health evaluation

In [23]:
from ibm_watson_openscale.utils.gen_ai_batch_utility import run_monitor_evaluation

mhm_run_id = run_monitor_evaluation(
    wos_client=wos_client,
    monitor_instance_id=mhm_monitor_instance_id,
    monitor_name="Model Health",
)


Triggering Model Health evaluation run...


Model Health run ID: 59de4a40-8e71-4990-8e1d-b5a2186e880f


  Model Health run status: running


  Model Health run status: running


  Model Health run status: running


...

  Model Health run status: running


  Model Health run status: running


  Model Health run status: running


  Model Health run status: running


  Model Health run status: finished
✅ Model Health evaluation run finished.


### 10-C. Display Model Health metrics

In [24]:
wos_client.monitor_instances.show_metrics(
    monitor_instance_id=mhm_monitor_instance_id
)

2026-10-13 16:22:47.276114+00:00,minimum_api_latency,01a13612-d8ec-7d15-b224-c2ab3fd4beeb,300.0,None,None,[],model_health,01a135d6-bf53-764f-8d3e-83f57ac5dd2a,59de4a40-8e71-4990-8e1d-b5a2186e880f,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:22:47.276114+00:00,average_api_latency,01a13612-d8ec-7d15-b224-c2ab3fd4beeb,1399.409,None,None,[],model_health,01a135d6-bf53-764f-8d3e-83f57ac5dd2a,59de4a40-8e71-4990-8e1d-b5a2186e880f,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:22:47.276114+00:00,median_api_latency,01a13612-d8ec-7d15-b224-c2ab3fd4beeb,1400.0,None,None,[],model_health,01a135d6-bf53-764f-8d3e-83f57ac5dd2a,59de4a40-8e71-4990-8e1d-b5a2186e880f,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:22:47.276114+00:00,maximum_api_latency,01a13612-d8ec-7d15-b224-c2ab3fd4beeb,2500.0,None,None,[],model_health,01a135d6-bf53-764f-8d3e-83f57ac5dd2a,59de4a40-8e71-4990-8e1d-b5a2186e880f,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:22:47.276114+00:00,minimum_api_throughput,01a13612-d8ec-7d15-b224-c2ab3fd4beeb,0.4,None,None,[],model_health,01a135d6-bf53-764f-8d3e-83f57ac5dd2a,59de4a40-8e71-4990-8e1d-b5a2186e880f,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:22:47.276114+00:00,average_api_throughput,01a13612-d8ec-7d15-b224-c2ab3fd4beeb,0.7145873722407101,None,None,[],model_health,01a135d6-bf53-764f-8d3e-83f57ac5dd2a,59de4a40-8e71-4990-8e1d-b5a2186e880f,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:22:47.276114+00:00,median_api_throughput,01a13612-d8ec-7d15-b224-c2ab3fd4beeb,0.7142857142857143,None,None,[],model_health,01a135d6-bf53-764f-8d3e-83f57ac5dd2a,59de4a40-8e71-4990-8e1d-b5a2186e880f,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:22:47.276114+00:00,maximum_api_throughput,01a13612-d8ec-7d15-b224-c2ab3fd4beeb,3.3333333333333335,None,None,[],model_health,01a135d6-bf53-764f-8d3e-83f57ac5dd2a,59de4a40-8e71-4990-8e1d-b5a2186e880f,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:22:47.276114+00:00,minimum_record_latency,01a13612-d8ec-7d15-b224-c2ab3fd4beeb,300.0,None,None,[],model_health,01a135d6-bf53-764f-8d3e-83f57ac5dd2a,59de4a40-8e71-4990-8e1d-b5a2186e880f,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:22:47.276114+00:00,average_record_latency,01a13612-d8ec-7d15-b224-c2ab3fd4beeb,1399.409,None,None,[],model_health,01a135d6-bf53-764f-8d3e-83f57ac5dd2a,59de4a40-8e71-4990-8e1d-b5a2186e880f,subscription,01a135d6-bcfe-7eff-9b96-a26159098581


Note: First 10 records were displayed.


<a id="11-drift-v2"></a>
## 11. Drift v2 Monitoring

Confirms the monitor is `active`, triggers an on-demand evaluation run, polls until
the run finishes, and displays metrics.

### 11-A. Re-check Drift v2 monitor is active

In [25]:
from ibm_watson_openscale.utils.gen_ai_batch_utility import wait_for_monitor_active

if not drift_v2_monitor_instance_id:
    print("⚠️  drift_v2_monitor_instance_id is not set — skipping Drift v2.")
else:
    wait_for_monitor_active(
        wos_client=wos_client,
        monitor_instance_id=drift_v2_monitor_instance_id,
        monitor_name="drift_v2",
    )


Waiting for 'drift_v2' (01a135d6-c5ea-717a-a1a9-9442f1c57520) to become active ...


  [21:57:31] attempt  1/30 — drift_v2: active
  ✅ drift_v2 is active.


### 11-B. Run Drift v2 evaluation

In [26]:
from ibm_watson_openscale.utils.gen_ai_batch_utility import run_monitor_evaluation

drift_run_id = run_monitor_evaluation(
    wos_client=wos_client,
    monitor_instance_id=drift_v2_monitor_instance_id,
    monitor_name="Drift v2",
)


Triggering Drift v2 evaluation run...


Drift v2 run ID: d1c8e3c2-b6e1-4c8e-a51f-18d6267bdbee


  Drift v2 run status: running


  Drift v2 run status: running


  Drift v2 run status: running


...

  Drift v2 run status: running


  Drift v2 run status: running


  Drift v2 run status: running


  Drift v2 run status: running


  Drift v2 run status: finished
✅ Drift v2 evaluation run finished.


### 11-C. Display Drift v2 metrics

In [27]:
wos_client.monitor_instances.show_metrics(
    monitor_instance_id=drift_v2_monitor_instance_id
)


2026-10-13 16:29:45.849021+00:00,input_metadata_drift_score,01a13619-3bf9-7f49-bd71-08a7bdcea67f,0.0037,None,None,"['algorithm_used:total_variation', 'computed_on:payload', 'field_type:input_meta', 'field_name:wos_input_character_count__']",drift_v2,01a135d6-c5ea-717a-a1a9-9442f1c57520,d1c8e3c2-b6e1-4c8e-a51f-18d6267bdbee,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:29:45.849021+00:00,records_processed,01a13619-3bf9-7f49-bd71-08a7bdcea67f,10000.0,None,None,"['algorithm_used:total_variation', 'computed_on:payload', 'field_type:input_meta', 'field_name:wos_input_character_count__']",drift_v2,01a135d6-c5ea-717a-a1a9-9442f1c57520,d1c8e3c2-b6e1-4c8e-a51f-18d6267bdbee,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:29:45.849021+00:00,input_metadata_drift_score,01a13619-3bf9-7f49-bd71-08a7bdcea67f,0.0057,None,None,"['algorithm_used:overlap_coefficient', 'computed_on:payload', 'field_type:input_meta', 'field_name:wos_input_character_count__']",drift_v2,01a135d6-c5ea-717a-a1a9-9442f1c57520,d1c8e3c2-b6e1-4c8e-a51f-18d6267bdbee,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:29:45.849021+00:00,records_processed,01a13619-3bf9-7f49-bd71-08a7bdcea67f,10000.0,None,None,"['algorithm_used:overlap_coefficient', 'computed_on:payload', 'field_type:input_meta', 'field_name:wos_input_character_count__']",drift_v2,01a135d6-c5ea-717a-a1a9-9442f1c57520,d1c8e3c2-b6e1-4c8e-a51f-18d6267bdbee,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:29:48.549017+00:00,input_metadata_drift_score,01a13619-4685-7c75-85cd-04e1bbb8962c,0.0027,None,None,"['algorithm_used:total_variation', 'computed_on:payload', 'field_type:input_meta', 'field_name:wos_input_word_count__']",drift_v2,01a135d6-c5ea-717a-a1a9-9442f1c57520,d1c8e3c2-b6e1-4c8e-a51f-18d6267bdbee,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:29:48.549017+00:00,records_processed,01a13619-4685-7c75-85cd-04e1bbb8962c,10000.0,None,None,"['algorithm_used:total_variation', 'computed_on:payload', 'field_type:input_meta', 'field_name:wos_input_word_count__']",drift_v2,01a135d6-c5ea-717a-a1a9-9442f1c57520,d1c8e3c2-b6e1-4c8e-a51f-18d6267bdbee,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:29:48.549017+00:00,input_metadata_drift_score,01a13619-4685-7c75-85cd-04e1bbb8962c,0.0047,None,None,"['algorithm_used:overlap_coefficient', 'computed_on:payload', 'field_type:input_meta', 'field_name:wos_input_word_count__']",drift_v2,01a135d6-c5ea-717a-a1a9-9442f1c57520,d1c8e3c2-b6e1-4c8e-a51f-18d6267bdbee,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:29:48.549017+00:00,records_processed,01a13619-4685-7c75-85cd-04e1bbb8962c,10000.0,None,None,"['algorithm_used:overlap_coefficient', 'computed_on:payload', 'field_type:input_meta', 'field_name:wos_input_word_count__']",drift_v2,01a135d6-c5ea-717a-a1a9-9442f1c57520,d1c8e3c2-b6e1-4c8e-a51f-18d6267bdbee,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:29:49.448653+00:00,input_metadata_drift_score,01a13619-4a08-70e2-816f-e28e1a8fa6d5,0.004,None,None,"['algorithm_used:total_variation', 'computed_on:payload', 'field_type:input_meta', 'field_name:wos_input_sentence_count__']",drift_v2,01a135d6-c5ea-717a-a1a9-9442f1c57520,d1c8e3c2-b6e1-4c8e-a51f-18d6267bdbee,subscription,01a135d6-bcfe-7eff-9b96-a26159098581
2026-10-13 16:29:49.448653+00:00,records_processed,01a13619-4a08-70e2-816f-e28e1a8fa6d5,10000.0,None,None,"['algorithm_used:total_variation', 'computed_on:payload', 'field_type:input_meta', 'field_name:wos_input_sentence_count__']",drift_v2,01a135d6-c5ea-717a-a1a9-9442f1c57520,d1c8e3c2-b6e1-4c8e-a51f-18d6267bdbee,subscription,01a135d6-bcfe-7eff-9b96-a26159098581


Note: First 10 records were displayed.


## Congratulations!

You have successfully completed the end-to-end GenAI Batch monitoring workflow with JDBC
storage. Your environment is now fully operational:

- ✅ A **Detached Prompt Template Asset (DPTA)** created directly in the deployment space
- ✅ **JDBC storage** and **watsonx.data Spark** registered as Integrated Systems
- ✅ A **batch subscription** created with landing, curated, and quarantine tables
- ✅ **Drift v2** baseline archive downloaded from GitHub and supplied at batch setup
- ✅ All monitor instances confirmed active
- ✅ Scoring data downloaded from GitHub, loaded into the landing table, and curated
- ✅ **Drift v2** and **Model Health** on-demand evaluation runs completed
- ✅ Metrics displayed

You can re-trigger evaluation runs on demand at any time, or wait for the scheduled
runs to execute automatically. For further details visit the
[Watson OpenScale Dashboard](https://url-to-your-cp4d-cluster/aiopenscale).
